In [51]:
import librosa
import numpy as np
import soundfile as sf
import pandas as pd
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

In [52]:
def extract_mfcc(file_path):
    y, sr=librosa.load(file_path , sr=16000, mono=True)
    mfcc=librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc=mfcc.T

    max_len=300
    if mfcc.shape[0] < max_len:
        pad=np.zeros((max_len-mfcc.shape[0], 40))
        mfcc=np.vstack((mfcc, pad))
    else:
        mfcc=mfcc[:max_len, :]
    return mfcc        

In [53]:
df_train = pd.read_csv("C:/MELD.Raw/train/train_sent_emo.csv")
df_test = pd.read_csv("C:/MELD.Raw/test/test_sent_emo.csv")
df_dev = pd.read_csv("C:/MELD.Raw/dev/dev_sent_emo.csv")

label_map = {
    "neutral": 0,
    "joy": 1,
    "sadness": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5,
    "disgust": 6
}

df_train['label'] = df_train['Emotion'].map(label_map)
df_test['label'] = df_test['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)

In [54]:
def load_audio_data(df,audio_folder):
    features = []
    labels = []
    skipped= 0

    for index, row in df.iterrows():
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']
        filename=f"dia{dia_id}_utt{utt_id}.wav"
        full_path=os.path.join(audio_folder, filename)

        if os.path.exists(full_path):
            mfcc = extract_mfcc(full_path)
            features.append(mfcc)
            labels.append(row['label'])
        else:
            skipped +=1
    print(f"Found {len(features)} audio files, skipped {skipped} files.")
    return features, labels        

In [55]:
import numpy as np

train_features = np.load("C:/MELD.Raw/Audio/train_features.npy")
train_labels = np.load("C:/MELD.Raw/Audio/train_labels.npy")

dev_features = np.load("C:/MELD.Raw/Audio/dev_features.npy")
dev_labels = np.load("C:/MELD.Raw/Audio/dev_labels.npy")

test_features = np.load("C:/MELD.Raw/Audio/test_features.npy")
test_labels = np.load("C:/MELD.Raw/Audio/test_labels.npy")

print(train_features.shape, dev_features.shape, test_features.shape)

(9988, 300, 40) (1108, 300, 40) (2610, 300, 40)


In [56]:
class AudioDataSet(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return{
            'features': feature,
            'labels': label
        }

In [57]:
mean = train_features.mean()
std = train_features.std()

train_features = (train_features - mean) / std
dev_features = (dev_features - mean) / std
test_features = (test_features - mean) / std

train_dataset = AudioDataSet(train_features, train_labels)
dev_dataset = AudioDataSet(dev_features, dev_labels)
test_dataset = AudioDataSet(test_features, test_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Done! Shapes:")
print(train_features.shape, dev_features.shape, test_features.shape)

Done! Shapes:
(9988, 300, 40) (1108, 300, 40) (2610, 300, 40)


In [58]:
len(train_features), len(dev_features), len(test_features)

(9988, 1108, 2610)

In [59]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(BiLSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

In [71]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMModel(input_size=40, hidden_size=256, num_layers=2, num_classes=7).to(device)
model=model.to(device)
print(f"Using device: {device}")
print(model)

Using device: cuda
BiLSTMModel(
  (lstm): LSTM(40, 256, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=512, out_features=7, bias=True)
)


In [72]:
y=train_labels
class_weight=compute_class_weight(class_weight='balanced', classes=np.array([0,1,2,3,4,5,6]), y=y)
class_weight=torch.tensor(class_weight,dtype=torch.float).to(device)

In [ ]:
total_steps = len(train_loader) * 50
warmup_steps = total_steps // 50

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
loss_fn=CrossEntropyLoss(weight=class_weight_tensor)

for epoch in range(50):
    print(f"Epoch {epoch+1}/{50}")
    model.train()
    total_loss = 0

    for batch in train_loader:
        features =batch["features"].to(device)
        labels =batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

avg_loss = total_loss / len(train_loader)
print(f"Average Loss: {avg_loss:.4f}")        

C:\Users\acer\AppData\Local\Temp\ipykernel_4156\2264732935.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)


Epoch 1/50
Epoch 2/50
Epoch 3/50
Epoch 4/50
Epoch 5/50
Epoch 6/50


In [ ]:
torch.save(model.state_dict(), "audio_model_v2.pth")
print("Model saved!")

TypeError: Module._save_to_state_dict() missing 2 required positional arguments: 'prefix' and 'keep_vars'

In [ ]:
# model.eval()
# all_preds = []
# all_labels = []

# with torch.no_grad():
#     for batch in test_loader:
#         features = batch["features"].to(device)
#         labels = batch["labels"].to(device)
#         outputs = model(features)
#         preds=torch.argmax(outputs, dim=1)
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(labels.cpu().numpy())

# from sklearn.metrics import classification_report

# target_names = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust']
# print(classification_report(all_labels, all_preds, target_names=target_names))


              precision    recall  f1-score   support

     neutral       0.47      0.09      0.15      1256
         joy       0.13      0.16      0.14       402
     sadness       0.11      0.05      0.07       208
       anger       0.12      0.30      0.17       345
        fear       0.01      0.06      0.02        50
    surprise       0.16      0.07      0.10       281
     disgust       0.03      0.18      0.05        68

    accuracy                           0.13      2610
   macro avg       0.15      0.13      0.10      2610
weighted avg       0.29      0.13      0.14      2610

